In [ ]:
import polars as pl
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from settings import load_settings

from capstone import construct_final_dataset as fd

In [ ]:
sns.set_theme(style="ticks")

In [ ]:
SETTINGS = load_settings()
ALL_PREDICTOR_VARIABLES = [
    fd.P_BINARY_ARTIFACTS,
    fd.P_BRANCH_PROTECTION,
    fd.P_CI_TESTS,
    fd.P_CII_BEST_PRACTICES,
    fd.P_CODE_REVIEW,
    fd.P_CONTRIBUTORS,
    fd.P_DANGEROUS_WORKFLOW,
    fd.P_DEPENDENCY_UPDATE_TOOL,
    fd.P_FUZZING,
    fd.P_LICENSE,
    fd.P_MAINTAINED,
    fd.P_PACKAGING,
    fd.P_PINNED_DEPENDENCIES,
    fd.P_SAST,
    fd.P_SECURITY_POLICY,
    fd.P_SIGNED_RELEASES,
    fd.P_TOKEN_PERMISSIONS,
]
FEATURE_COLUMNS = fd.ALL_CONTROL_VARIABLES + ALL_PREDICTOR_VARIABLES

In [ ]:
final_df = pl.read_parquet(SETTINGS.final_dataset_path)

In [ ]:
final_df.shape

In [ ]:
final_df.head()

## Spearman Correlation Matrix

In [ ]:
def plot_spearman_correlation_matrix(data: pl.DataFrame, feature_columns: list[str] = None):
    """Plots a Spearman correlation matrix using Seaborn.

    Parameters:
    - data: A Polars DataFrame containing the data.
    - feature_columns: A list of column names to include in the correlation matrix.
    """
    df = data.select(feature_columns).to_pandas()

    # Sort alphabetically and strip 'github_repo_' and 'package_' prefixes from column names
    df.columns = sorted([
        col.removeprefix("github_repo_").removeprefix("package_")
        for col in df.columns
    ])

    corr_matrix = df.corr(method='spearman')

    # Create a mask for the upper triangle
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    fig, ax = plt.subplots(figsize=(18, 14))

    sns.heatmap(
        corr_matrix, 
        mask=mask,
        annot=True, 
        vmin=-1,
        center=0,  
        vmax=1, 
        square=True,
        cmap='coolwarm',
        cbar=True,                # Show color bar
        fmt='.2f',
        linewidths=0,             # No global grid lines
        cbar_kws={'shrink': .8},  # Adjust color bar size
        #annot_kws={"size": 8}
        ax=ax,
    )

    # Draw lines only on the visible (lower triangle) cells
    n = len(corr_matrix)
    for i in range(n):
        for j in range(i):
            ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='black', linewidth=0.5))

    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)  
    #plt.title('Spearman Correlation Matrix Heatmap of Predictor Variables', fontsize=16)
    plt.tight_layout()
    plt.savefig(f"{SETTINGS.visualizations_path}/spearman_correlation_matrix.pdf", bbox_inches="tight") 
    plt.show()


In [ ]:
plot_spearman_correlation_matrix(final_df, feature_columns=FEATURE_COLUMNS)